<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/03-watermarking-late-data.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 14.3 — Watermarking: bounded state, and a concrete drop rule

Build a synthetic "late arrivals" stream by jittering `event_time`
backward from `rate`'s own timestamp, window + watermark it, and read
`numRowsDroppedByWatermark` straight off the query's progress metrics
-- the drop rule from the lesson, observed as a number, not asserted.

In [ ]:
import os, sys, shutil, time, json
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.3-watermarking")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## A stream with deliberately out-of-order event time

`rate`'s own `timestamp` always arrives in order (processing time).
We manufacture `event_time` by subtracting a random jitter -- some
rows will look "late" relative to others already seen.

In [ ]:
jittered = (spark.readStream.format("rate").option("rowsPerSecond", 10).load()
            .withColumn("jitter_seconds", (F.rand() * 8).cast("int"))
            .withColumn("event_time",
                        col("timestamp") - F.expr("INTERVAL 1 SECONDS") * col("jitter_seconds")))
# event_time is now UP TO 8 SECONDS behind processing time, randomly per row --
# exactly the "arrives late relative to event time" scenario 14.3 describes

## Window + watermark: the policy in code

In [ ]:
windowed = (jittered
            .withWatermark("event_time", "5 seconds")   # tolerate up to 5s of lateness
            .groupBy(F.window("event_time", "3 seconds"))
            .count())

wm_query = (windowed.writeStream
            .outputMode("update")            # legal now -- see 14.2's table
            .format("memory").queryName("watermark_demo")
            .start())

time.sleep(10)
print("windowed counts so far (some windows may still be open):")
spark.sql("SELECT window.start, window.end, count FROM watermark_demo ORDER BY window.start").show(20, truncate=False)

## The drop rule, as a number: `numRowsDroppedByWatermark`

Straight off `StreamingQueryProgress` -- this is the metric 14.3
promises exists for measuring, not just asserting, late-data loss.

In [ ]:
time.sleep(5)
progress = wm_query.recentProgress
total_dropped = 0
for p in progress:
    for op in p.get("stateOperators", []):
        total_dropped += op.get("numRowsDroppedByWatermark", 0)
print(f"total rows dropped by the watermark across {len(progress)} recent batches: {total_dropped}")
print("(some jittered rows arrived 'later', relative to the advancing watermark, "
      "than the 5-second tolerance allows -- this is 14.3's trade-off, made visible)")

wm_query.stop()

## `append` mode now works -- the watermark makes it legal

Compare: 14.2 showed `append` + aggregation rejected with NO
watermark. Same shape, watermark added, now legal.

In [ ]:
append_windowed_q = (windowed.writeStream
                     .outputMode("append")   # only emits once a window is PROVABLY closed
                     .format("memory").queryName("watermark_append_demo")
                     .start())

time.sleep(10)
print("append mode -- only CLOSED windows appear (may lag behind 'update' mode's live view):")
spark.sql("SELECT window.start, window.end, count FROM watermark_append_demo ORDER BY window.start").show(truncate=False)
append_windowed_q.stop()

In [ ]:
spark.stop()

## Exercises

1. Widen the watermark from `"5 seconds"` to `"15 seconds"` and
   re-run. Does `numRowsDroppedByWatermark` go up or down? What's the
   cost you're paying for that improvement (reread the lesson's
   memory trade-off)?
2. Narrow the watermark to `"1 second"` (aggressive). Watch the drop
   count. At what point does the watermark become so tight that most
   jittered rows are dropped?
3. Compare the `update`-mode table and the `append`-mode table for
   the SAME windows. Which one shows a window's count changing over
   time, and which shows it only once, final? Connect this to 14.2's
   mode semantics.
4. Add a genuinely bad row: one jittered timestamp set FAR in the
   future (`event_time = current_timestamp() + INTERVAL 1 HOUR` for
   one synthetic row). What happens to the watermark and to
   subsequent "normal" rows' windows? (14.3's future-timestamp danger,
   reproduced.)
5. Switch the window from tumbling (`F.window("event_time", "3
   seconds")`) to sliding (`F.window("event_time", "3 seconds", "1
   second")`) and observe how many windows a single event now falls
   into. How does this change the shape of the output table?

In [ ]:
# your exercise code here